# scANVI Integration with HCA Lung Reference

Same in-house (fused, parental, resistant) vs. public HCA lung reference setup as the
scVI notebook, but using **scANVI** instead of plain scVI.

scANVI is scVI's semi-supervised sibling: it uses the HCA `cell_type` labels as a
supervisory signal during training (in-house cells are marked `"Unknown"`), which
tends to produce cleaner reference mapping than an unsupervised joint embedding —
and it comes with a built-in query-to-reference label-prediction step, which answers
"which cells overlap, and with what" directly instead of needing a separate kNN
lookup. This is the same underlying model family used by `scArches` for atlas-scale
reference mapping.

**Workflow:**
1. Load in-house 10x data
2. Download HCA lung data via CellxGene Census
3. Verify raw counts in HCA data
4. Tag + concatenate (preserving `cell_type`/`assay`)
5. QC filter
6. HVG selection with `batch_key='source'`
7. Train scVI, then initialize + fine-tune scANVI from it (`scvi.model.SCANVI.from_scvi_model`)
8. Predict cell type + confidence per cell; embedding, clustering, visualization
9. Mixing diagnostics (which cells overlap, and with what)
10. DE between in-house sample groups (restricted to in-house cells)
11. Cell type composition / in-house-only UMAP

## Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scvi
import cellxgene_census  # pip install cellxgene-census
import matplotlib.pyplot as plt

sc.set_figure_params(figsize=(5, 5))
%config InlineBackend.print_figure_kwargs={'facecolor': 'w'}
%config InlineBackend.figure_format='retina'

## 1. Load In-House Data

In [ ]:
adata_fus = sc.read_10x_mtx('../fused/', var_names='gene_symbols', cache=True)
adata_fus.obs['sample'] = 'fused'
adata_fus.obs.index = [f'F_{bc}' for bc in adata_fus.obs.index]

adata_par = sc.read_10x_mtx('../parental/', var_names='gene_symbols', cache=True)
adata_par.obs['sample'] = 'parental'
adata_par.obs.index = [f'P_{bc}' for bc in adata_par.obs.index]

adata_res = sc.read_10x_mtx('../resistant/', var_names='gene_symbols', cache=True)
adata_res.obs['sample'] = 'resistant'
adata_res.obs.index = [f'R_{bc}' for bc in adata_res.obs.index]

for adata in [adata_fus, adata_par, adata_res]:
    adata.var_names_make_unique()
    adata.obs['source'] = 'in_house'

print(adata_fus)
print(adata_par)
print(adata_res)

## 2. Download HCA Lung Reference via CellxGene Census

Streams lung cells directly — no need to download the full atlas.
Set `census_version` to a pinned release for reproducibility; check available versions at
https://chanzuckerberg.github.io/cellxgene-census/cellxgene_census_docsite_data_release_info.html

In [ ]:
CENSUS_VERSION = '2025-11-08'  # pin for reproducibility

census = cellxgene_census.open_soma(census_version=CENSUS_VERSION)

adata_hca = cellxgene_census.get_anndata(
    census,
    organism='Homo sapiens',
    obs_value_filter="tissue_general == 'lung' and is_primary_data == True",
)

census.close()

print(adata_hca)
print('\nHCA obs columns:', adata_hca.obs.columns.tolist())

## 3. Verify HCA Raw Counts

scVI requires raw integer counts. CellxGene Census stores raw counts in `.X` by default, but confirm here.

In [ ]:
import scipy.sparse as sp

sample_vals = adata_hca.X[:10, :10]
if sp.issparse(sample_vals):
    sample_vals = sample_vals.toarray()

print('Sample HCA X values (should be non-negative integers):')
print(sample_vals)
print('\nAll integer:', np.all(sample_vals == np.floor(sample_vals)))

## 4. Tag Metadata — Preserve `cell_type` and `assay` Across the Join

**Fix vs. the original notebook:** `ad.concat(..., join='inner')` inner-joins `.obs`
columns too, not just genes. Because `cell_type` and `assay` only existed on the HCA
side, they were silently dropped during concatenation — which is why the original
"Cell type (HCA annotations)" plot printed `No cell_type column found`.

The fix is to make sure every column we want to keep exists (even as a placeholder)
on **both** sides before concatenating. We also check `assay` here: the Census lung
pull mixes multiple sequencing technologies (e.g. 10x 3' v2/v3, Smart-seq2), and that
was never modeled as a covariate in the original run — it's an unmodeled technical
gap sitting on top of the in-house-vs-HCA batch effect.

In [ ]:
adata_hca.obs['sample'] = 'HCA_lung'
adata_hca.obs['source'] = 'public_HCA'

# Make barcodes unique to avoid collisions with in-house barcodes
adata_hca.obs.index = [f'HCA_{bc}' for bc in adata_hca.obs.index]

print('HCA assay breakdown (technical heterogeneity hiding inside the "public_HCA" batch):')
print(adata_hca.obs['assay'].value_counts())

print(adata_hca)

In [ ]:
# Concatenate in-house samples first (same gene space)
adata_inhouse = ad.concat([adata_fus, adata_par, adata_res], join='outer')

# Placeholders so these columns survive the inner join on .obs below.
# 'cell_type' stays a real label for HCA cells and 'in_house' for ours, so it's
# usable later both as ground truth (HCA) and as a marker of "not yet identified" (ours).
adata_inhouse.obs['cell_type'] = 'in_house'
adata_inhouse.obs['assay'] = "10x (in-house)"  # update to your actual chemistry (e.g. "10x 3' v3") if known

adata_combined = ad.concat(
    [adata_inhouse, adata_hca],
    join='inner',   # keeps only genes shared between in-house and HCA
    merge='same',
)

print(adata_combined)
print(f'\nShared genes: {adata_combined.n_vars}')
print(f'Source breakdown:')
print(adata_combined.obs['source'].value_counts())
print(f'\ncell_type preserved:', 'cell_type' in adata_combined.obs.columns)
print(f'assay preserved:', 'assay' in adata_combined.obs.columns)

## 5. QC Filtering

In [ ]:
sc.pp.filter_cells(adata_combined, min_genes=200)
sc.pp.filter_genes(adata_combined, min_cells=3)

adata_combined.var['mt'] = adata_combined.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(
    adata_combined, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True
)

# Visualize QC metrics split by source before filtering
sc.pl.violin(
    adata_combined,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    groupby='source',
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_combined_filtered = adata_combined[adata_combined.obs.total_counts < 75000, :]
adata_combined_filtered = adata_combined_filtered[adata_combined_filtered.obs.n_genes > 1000, :]
adata_combined_filtered = adata_combined_filtered[adata_combined_filtered.obs.pct_counts_mt < 40, :]

print(adata_combined_filtered)
print(adata_combined_filtered.obs['source'].value_counts())

## 6. Normalize, Log-Transform, and Select HVGs

HVGs are selected with `batch_key='source'` so genes are ranked by variability within each source
independently, then combined. This avoids selecting genes that vary only because of source differences.

In [ ]:
adata_combined_filtered = adata_combined_filtered.copy()  # materialize view
adata_combined_filtered.layers['counts'] = adata_combined_filtered.X.copy()

sc.pp.normalize_total(adata_combined_filtered, target_sum=1e6)
sc.pp.log1p(adata_combined_filtered)
adata_combined_filtered.raw = adata_combined_filtered

sc.pp.highly_variable_genes(
    adata_combined_filtered,
    n_top_genes=3000,
    subset=True,
    layer='counts',
    flavor='seurat_v3',
    batch_key='source',      # rank HVGs within each source independently
)

print(f'HVGs selected: {adata_combined_filtered.n_vars}')

## 7. Base scVI Model

scvi-tools' recommended way to train scANVI is to first train a plain scVI model,
then initialize scANVI from it (`SCANVI.from_scvi_model`) and fine-tune for a much
smaller number of epochs. `n_latent=30` and the epoch count follow the same reasoning
as the fixed scVI notebook — 128 latent dims / 1000 flat epochs were not justified by
dataset size or a convergence check.

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_combined_filtered,
    layer='counts',
    batch_key='source',
    categorical_covariate_keys=['sample', 'assay'],
)

In [ ]:
n_obs = adata_combined_filtered.n_obs
heuristic_max_epochs = min(round((20000 / n_obs) * 400), 400)
print(f'n_cells={n_obs}, scvi-tools default max_epochs heuristic ~= {heuristic_max_epochs}')

scvi_model = scvi.model.SCVI(adata_combined_filtered, n_latent=30)
scvi_model.train(
    accelerator='mps',
    max_epochs=heuristic_max_epochs,
    early_stopping=True,
    early_stopping_patience=20,
)

## 8. scANVI Setup and Fine-Tuning

- `labels_key`: HCA cells keep their real `cell_type`; every in-house cell is
  labeled `'Unknown'`, so scANVI treats it as a query to be classified against the
  HCA-derived label space.
- Fine-tuning needs far fewer epochs than the base scVI model — scvi-tools' own
  tutorials use ~20 by default.

In [ ]:
CELL_TYPE_LABEL_KEY = 'cell_type_scanvi'
adata_combined_filtered.obs[CELL_TYPE_LABEL_KEY] = adata_combined_filtered.obs['cell_type'].astype(str)
adata_combined_filtered.obs.loc[
    adata_combined_filtered.obs['source'] == 'in_house', CELL_TYPE_LABEL_KEY
] = 'Unknown'

print(adata_combined_filtered.obs[CELL_TYPE_LABEL_KEY].value_counts().head(20))

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model,
    adata=adata_combined_filtered,
    labels_key=CELL_TYPE_LABEL_KEY,
    unlabeled_category='Unknown',
)

scanvi_max_epochs = min(20, heuristic_max_epochs)
scanvi_model.train(
    accelerator='mps',
    max_epochs=scanvi_max_epochs,
    early_stopping=True,
)

In [ ]:
train_elbo = scanvi_model.history['elbo_train']
val_elbo = scanvi_model.history['elbo_validation']

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(train_elbo.index, train_elbo.iloc[:, 0], label='train')
ax.plot(val_elbo.index, val_elbo.iloc[:, 0], label='validation')
ax.set_xlabel('epoch')
ax.set_ylabel('ELBO')
ax.legend()
ax.set_title('scANVI fine-tuning curves')
plt.show()

In [ ]:
# Save model so you don't have to retrain
scanvi_model.save('scanvi_HCA_integration_model/', overwrite=True)

## 9. Latent Representation, Predicted Labels, and Clustering

`scanvi_model.predict()` gives a hard label per cell; `predict(soft=True)` gives the
full per-class probability distribution, whose max is a natural per-cell confidence /
uncertainty score — directly usable to say which in-house cells are confidently
mapped onto a known HCA cell type (overlapping) vs. ambiguous (not overlapping,
possibly a genuinely novel state).

In [ ]:
SCANVI_LATENT_KEY = 'X_scANVI'
adata_combined_filtered.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation()

adata_combined_filtered.obs['scanvi_predicted_cell_type'] = scanvi_model.predict()

soft_preds = scanvi_model.predict(soft=True)
adata_combined_filtered.obs['scanvi_prediction_confidence'] = soft_preds.max(axis=1).values

sc.pp.neighbors(adata_combined_filtered, use_rep=SCANVI_LATENT_KEY)

SCANVI_CLUSTERS_KEY = 'leiden_scANVI'
sc.tl.leiden(
    adata_combined_filtered,
    key_added=SCANVI_CLUSTERS_KEY,
    resolution=0.5,
    flavor='igraph',
    n_iterations=2,
    directed=False,
)

sc.tl.umap(adata_combined_filtered, min_dist=0.3)

## 10. Visualization

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['source'],
    frameon=False,
    title='Source (integration check)',
)

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['sample'],
    frameon=False,
    groups=['fused', 'parental', 'resistant'],
    title='In-house samples',
)

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=[SCANVI_CLUSTERS_KEY],
    frameon=False,
    legend_loc='on data',
)

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['cell_type'],
    frameon=False,
    legend_fontsize=7,
    title='Cell type (HCA ground-truth annotations)',
)

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['scanvi_predicted_cell_type', 'scanvi_prediction_confidence'],
    frameon=False,
    legend_fontsize=7,
    ncols=2,
)

In [ ]:
inhouse_mask = adata_combined_filtered.obs['source'] == 'in_house'

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(
    adata_combined_filtered.obs.loc[inhouse_mask, 'scanvi_prediction_confidence'],
    bins=40,
)
ax.set_xlabel('scANVI prediction confidence (max class probability)')
ax.set_ylabel('# in-house cells')
ax.set_title('How confidently does scANVI map in-house cells onto an HCA cell type?')
plt.show()

CONFIDENCE_THRESHOLD = 0.5  # tune based on the histogram above
adata_combined_filtered.obs['well_integrated'] = (
    adata_combined_filtered.obs['scanvi_prediction_confidence'] >= CONFIDENCE_THRESHOLD
)

print('Predicted cell type for confidently-mapped ("overlapping") in-house cells:')
confident_inhouse = inhouse_mask & adata_combined_filtered.obs['well_integrated']
print(
    adata_combined_filtered.obs.loc[confident_inhouse, 'scanvi_predicted_cell_type']
    .value_counts()
    .head(15)
)

## Per-Cell and Per-Cluster Mixing Diagnostics

Two complementary views of "which cells actually overlap":

1. **Per-cell local mixing score** — for every cell, what fraction of its nearest
   neighbors (in the scVI/scANVI latent space) are `public_HCA`? Near the background
   HCA fraction ⇒ well mixed; near 0 ⇒ isolated in its own in-house neighborhood.
2. **Nearest-HCA cell type** — restricted to HCA neighbors only, what `cell_type`
   do they carry? This tells you *which* HCA cell type an in-house cell resembles,
   not just whether it's "near" the reference in general.

In [ ]:
conn = adata_combined_filtered.obsp['connectivities']
is_hca = (adata_combined_filtered.obs['source'] == 'public_HCA').values.astype(float)

neighbor_hca_frac = (
    np.asarray(conn.multiply(is_hca[None, :]).sum(axis=1)).ravel()
    / np.asarray((conn > 0).sum(axis=1)).ravel()
)
adata_combined_filtered.obs['hca_neighbor_frac'] = neighbor_hca_frac

background_hca_frac = is_hca.mean()
print(f'Background HCA fraction (expected value if fully mixed): {background_hca_frac:.3f}')

# A simple, tunable definition of "overlapping": at least as HCA-mixed as random chance
adata_combined_filtered.obs['well_integrated'] = (
    adata_combined_filtered.obs['hca_neighbor_frac'] >= background_hca_frac
)

inhouse_mask = adata_combined_filtered.obs['source'] == 'in_house'
print('\nIn-house cells flagged well_integrated:')
print(adata_combined_filtered.obs.loc[inhouse_mask, 'well_integrated'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for s in ['fused', 'parental', 'resistant']:
    vals = adata_combined_filtered.obs.loc[
        adata_combined_filtered.obs['sample'] == s, 'hca_neighbor_frac'
    ]
    ax.hist(vals, bins=40, alpha=0.5, label=s, density=True)
ax.axvline(background_hca_frac, color='k', linestyle='--', label='background HCA fraction')
ax.set_xlabel('Fraction of neighbors that are public_HCA')
ax.set_ylabel('Density')
ax.legend()
ax.set_title('Per-cell local mixing score, by in-house sample')
plt.show()

In [ ]:
sc.pl.umap(
    adata_combined_filtered,
    color=['hca_neighbor_frac'],
    frameon=False,
    color_map='viridis',
    title='Local mixing score (fraction of neighbors from HCA)',
)

In [ ]:
# Cluster-level view: source composition per leiden_scANVI cluster
composition = pd.crosstab(
    adata_combined_filtered.obs['leiden_scANVI'],
    adata_combined_filtered.obs['source'],
    normalize='index',
)
print('Fraction of each cluster from each source (compare to background HCA fraction above):')
print(composition.sort_values('public_HCA'))

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.stats import mode

latent = adata_combined_filtered.obsm['X_scANVI']
is_hca_mask = (adata_combined_filtered.obs['source'] == 'public_HCA').values
is_inhouse_mask = ~is_hca_mask

nn = NearestNeighbors(n_neighbors=15).fit(latent[is_hca_mask])
_, idx = nn.kneighbors(latent[is_inhouse_mask])

hca_cell_types = adata_combined_filtered.obs.loc[is_hca_mask, 'cell_type'].values
nn_types = hca_cell_types[idx]
maj = mode(nn_types, axis=1, keepdims=False)

inhouse_names = adata_combined_filtered.obs_names[is_inhouse_mask]
adata_combined_filtered.obs.loc[inhouse_names, 'nearest_hca_cell_type'] = maj.mode
adata_combined_filtered.obs.loc[inhouse_names, 'nearest_hca_cell_type_agreement'] = maj.count / nn_types.shape[1]

print('Which HCA cell types do well-integrated in-house cells resemble?')
well_mixed_inhouse = inhouse_names[
    adata_combined_filtered.obs.loc[inhouse_names, 'well_integrated'].values
]
print(
    adata_combined_filtered.obs.loc[well_mixed_inhouse, 'nearest_hca_cell_type']
    .value_counts()
    .head(15)
)

## 11. Differential Expression — In-House Samples Only

Restrict DE to in-house cells so we compare fused/parental/resistant against each other,
not against HCA lung cells. Uses the base scVI model (scANVI's DE relies on the
same generative model but is initialized from it).

In [ ]:
adata_inhouse_filtered = adata_combined_filtered[
    adata_combined_filtered.obs['source'] == 'in_house'
].copy()

print(adata_inhouse_filtered)
print(adata_inhouse_filtered.obs['sample'].value_counts())

In [ ]:
de_df = scanvi_model.differential_expression(
    adata=adata_inhouse_filtered,
    groupby='sample',
    mode='change',
)
de_df.head()

In [ ]:
markers = {}
cats = {'fused', 'resistant', 'parental'}
for c in cats:
    cid = f'{c} vs Rest'
    cell_type_df = de_df.loc[de_df.comparison == cid]
    cell_type_df = cell_type_df[cell_type_df.lfc_mean > 0]
    cell_type_df = cell_type_df[cell_type_df['bayes_factor'] > 3]
    cell_type_df = cell_type_df[cell_type_df['non_zeros_proportion1'] > 0.1]
    markers[c] = cell_type_df.index.tolist()[:3]

print('Top markers per sample:', markers)

In [ ]:
sc.tl.dendrogram(adata_inhouse_filtered, groupby='sample', use_rep='X_scANVI')

sc.pl.dotplot(
    adata_inhouse_filtered,
    markers,
    groupby='sample',
    dendrogram=True,
    color_map='Blues',
    swap_axes=True,
    use_raw=True,
    standard_scale='var',
)

## 12. Cell Type Composition of In-House Samples

Shows what fraction of each in-house sample was assigned to each predicted HCA cell type.

In [ ]:
composition = (
    adata_inhouse_filtered.obs
    .groupby(['sample', 'scanvi_predicted_cell_type'], observed=True)
    .size()
    .unstack(fill_value=0)
)
composition_norm = composition.div(composition.sum(axis=1), axis=0)

composition_norm.T.plot(
    kind='bar',
    stacked=False,
    figsize=(14, 5),
    title='scANVI-predicted cell type composition by in-house sample',
)
plt.ylabel('Fraction of cells')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 13. UMAP — In-House Cells Only

In [ ]:
sc.pl.umap(
    adata_inhouse_filtered,
    color=['sample', SCANVI_CLUSTERS_KEY, 'scanvi_prediction_confidence', 'well_integrated'],
    frameon=False,
    ncols=2,
)